# C1DB1B: Data Management for Decision-Making
## Workshop 2: Data Preparation, Hygiene & Pipeline Amalgamation
### Executive Decision Setting: Allocating the 500 Million SEK VGR Healthcare Investment Fund

---

### Executive Case Background & Pedagogical Overview
In **Module 1 (Gateway 1)**, you curated raw administrative census tables from Statistics Sweden (**SCB**) and established an audit-ready demographic baseline across **Västra Götalandsregionen (VGR)**. You quantified demographic momentum, dispersion ($\sigma, CV$), and aging intensity ratios ($80+$ cohorts).

In **Module 2 (Gateway 2)**, regional healthcare executives face a critical organizational challenge: **demographic momentum alone is insufficient to allocate 500 million SEK in healthcare capital investments**. An aging population in an affluent urban center with extensive transit infrastructure (e.g., Mölndal or Partille) requires vastly different capital intervention than an aging population in an isolated rural or archipelago community (e.g., Dals-Ed, Färgelanda, or Öckerö) characterized by severe transit bottlenecks and socioeconomic distress.

To support high-stakes executive decisions, you must build an amalgamated **decision-support data mart** by integrating multiple disparate regional data pipelines.

```
       ========================================================================================
                                     CONTINUOUS GATEWAY WORKFLOW (MODULE 2)
       ========================================================================================
       [ Gateway 1 Baseline ]          [ External Public Health & SCB Tables ]
        Demographics (SCB 2015-2025)     • Travel Times to Acute Care Facilities
        Total Pop & 80+ Age Cohorts      • Real SCB Socioeconomic Index (Table TAB6586)
        Aging Ratios & Growth Rates      • Median Municipal Incomes & Care Utilization
                     │                                      │
                     └───────────────────┬──────────────────┘
                                         ▼
                            [ Relational Pipeline Join ]
                             Key: `municipality_code` (1:1)
                             Zero Row Loss Audit (52 / 52)
                                         │
                                         ▼
                        [ Data Hygiene & Imputation Audit ]
                         Unmask SCB Missing Tokens (`..`, `-`, ` `)
                         Imputation Sensitivity: $\sigma_{mean}$ vs. $\sigma_{median}$
                                         │
                                         ▼
                     [ Group Portfolio Preprocessing Menu ]
                      Option A: Log Transform (Income Right Skew)
                      Option B: Robust Winsorization (Transit Outliers)
                      Option C: Standard Z-Score (Real SCB Socioeconomic Bell Curve)
                      Option D: Min-Max Feature Scaling (Care Utilization)
                                         │
                                         ▼
                         [ Validation Governance Firewall ]
                          80% Training Set  (41 Municipalities)
                          20% Sequestered   (11 Municipalities - Blind Hold-Out)
       ========================================================================================
```

---

### Learning Objectives:
1. **Relational Pipeline Amalgamation**: Master relational data objects (Scalars, Vectors, DataFrames) and execute primary-key joins (`municipality_code`) ensuring zero record loss.
2. **Interactive Public Sector API Ingestion**: Query live public REST endpoints (Statistics Sweden API v2) without API keys, parse multi-dimensional `JSON-stat 2.0` objects, and extract real sub-district indicators.
3. **Data Hygiene & Imputation Sensitivity**: Unmask administrative null markers (`..`, ` `, `-`) and evaluate variance compression ($\sigma_{mean} < \sigma_{median}$) and equity distortions under mean vs. median imputation.
4. **Preprocessing Portfolio Specialization**: Diagnose non-commensurate scales and implement tailored mathematical transformations (Log, Winsorization, Z-score, Min-Max).
5. **Validation Governance & Ethics**: Sequester an immutable 20% hold-out test set to prevent data leakage and model optimism before Module 3 predictive modeling.


---
## 🧱 Part 1: Foundations & Python Toolkit

### Decision Theory & Data Governance Grounding: Understanding Tabular Data Objects
Before performing data joins, a decision-support analyst must understand how data structures map to organizational reality:
1. **0D Scalars (Single Values)**: Fundamental quantitative parameters (e.g., total capital budget $B = 500{,}000{,}000\text{ SEK}$, regional average aging ratio $\bar{x} = 7.4\%$, test split ratio $\alpha = 0.20$).
2. **1D Vectors / Pandas Series**: Single organizational attributes tracked across entities (e.g., the array of travel times across all 52 municipalities).
3. **2D Tabular Matrices / Pandas DataFrames**: Multi-attribute registries where each row represents an administrative entity (**primary key**) and each column represents an orthogonal indicator.
4. **Dictionaries as Relational Mappings**: Key-value lookup tables used for schema harmonization, missing-code replacement maps, and foreign key translation.

In public healthcare governance, data integrity depends on **Primary Keys**: unique numeric identifiers (Statistics Sweden municipality codes `1401`–`1499`) that permanently identify administrative jurisdictions across census, clinical, and infrastructure datasets.


In [ ]:
# ==============================================================================
# PART 1: ENVIRONMENT SETUP & LIBRARY INITIALIZATION
# ==============================================================================

import os
import json
import urllib.request
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

# Visual layout and numeric formatting
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
pd.set_option('display.max_columns', 15)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', lambda x: f'{x:.3f}')

print(" Python Decision-Support Environment Successfully Initialized!")
print(f"Pandas Version: {pd.__version__} | NumPy Version: {np.__version__}")


---
## 🔗 Part 2: Pipeline Amalgamation (The "4+1" Join)

### Step 2.1: Extracting the Municipal Demographic Baseline
To establish our baseline from Module 1 (`VGR_Demographics_2015_2025.csv`), we aggregate single-year age cohorts into four core decision metrics for each municipality:
1. `total_pop_2025`: Total municipal population in 2025.
2. `pop_80plus_2025`: Total senior population aged 80 and older in 2025.
3. `aging_ratio_2025`: Senior dependency ratio $\left(\frac{\text{pop\_80plus\_2025}}{\text{total\_pop\_2025}} \times 100\right)$.
4. `growth_80plus_pct`: 10-year growth velocity of the senior population (2015–2025):
   $$\text{Growth}_{80+} = \frac{\text{pop\_80plus\_2025} - \text{pop\_80plus\_2015}}{\text{pop\_80plus\_2015}} \times 100$$


In [ ]:
# ==============================================================================
# STEP 2.1: EXTRACTING MUNICIPAL DEMOGRAPHIC BASELINE FROM MODULE 1
# ==============================================================================

demo_file = "VGR_Demographics_2015_2025.csv"
if not os.path.exists(demo_file):
    raise FileNotFoundError(f"Missing demographic baseline: {demo_file}. Run Module 1 first!")

df_demo_raw = pd.read_csv(demo_file)

# 1. Filter out county-level aggregate (region_code 14) and isolate total gender and individual age cohorts
muni_demo = df_demo_raw[
    (df_demo_raw['region_code'] != 14) & 
    (df_demo_raw['gender_label'] == 'totalt') & 
    (df_demo_raw['age_code'] != 'TOT')
].copy()
muni_demo['age'] = pd.to_numeric(muni_demo['age_code'], errors='coerce')

# 2. Extract total population across all age cohorts for 2025 and 2015
df_tot = muni_demo.groupby(['region_code', 'region_name'])[['pop_2025', 'pop_2015']].sum().reset_index()
df_tot = df_tot.rename(columns={
    'region_code': 'municipality_code',
    'region_name': 'municipality_name',
    'pop_2025': 'total_pop_2025',
    'pop_2015': 'total_pop_2015'
})

# 3. Aggregate senior cohorts aged 80 and older
df_80p = muni_demo[muni_demo['age'] >= 80].groupby('region_code')[['pop_2025', 'pop_2015']].sum().reset_index()
df_80p = df_80p.rename(columns={
    'region_code': 'municipality_code',
    'pop_2025': 'pop_80plus_2025',
    'pop_2015': 'pop_80plus_2015'
})

# 4. Merge demographic totals and compute strategic KPIs
df_baseline = pd.merge(df_tot, df_80p, on='municipality_code')
df_baseline['aging_ratio_2025'] = (df_baseline['pop_80plus_2025'] / df_baseline['total_pop_2025']) * 100.0
df_baseline['growth_80plus_pct'] = ((df_baseline['pop_80plus_2025'] - df_baseline['pop_80plus_2015']) / df_baseline['pop_80plus_2015']) * 100.0

print(f" Demographic baseline created successfully across {len(df_baseline)} municipalities!")
display(df_baseline[['municipality_code', 'municipality_name', 'total_pop_2025', 'aging_ratio_2025', 'growth_80plus_pct']].head(4))


---
### 🌐 Interactive Data Engineering Extension: Live Public Sector API Ingestion
#### Harvesting Open Data from Statistics Sweden (SCB) REST API v2 (No API Key Required!)

In modern health informatics and regional planning, analysts rarely rely solely on static CSV exports. Government agencies and statistical bureaus publish live REST endpoints.

**Statistics Sweden (Statistiska centralbyrån - SCB)** provides an open API v2 for Table **TAB6586** (*Socioekonomiskt index efter region och år*, Product `LE0105`):
- **Authentication**: Completely open public data — **no API key, registration, or token required**!
- **Data Standard**: Delivers multi-dimensional cubes in the international `JSON-stat 2.0` open format.
- **Geographic Granularity**: Reports statistics at the **RegSO (Regionala statistikområden)** level.
  - In Sweden, every municipality is partitioned into granular neighborhood sub-districts.
  - For example, **Borås (`1490`)** contains **47 RegSO areas** (`1490R001` to `1490R047`), including *Norrby*, *Hässleholmen*, *Sjöbo*, *Trandared*, *Bergdalen*, and *Dalsjöfors*.
- **Key Indicators in Table TAB6586**:
  1. `000008IT`: **Socioekonomiskt index** — A composite socioeconomic challenge index calculated by SCB as $\frac{V1 + V2 + V3}{3}$ (low income share, low education, and long-term economic assistance). Higher values indicate greater socioeconomic and health vulnerabilities.
  2. `000008IU`: **Områdestyp** — Socioeconomic area classification (Type 1 = *Areas with major challenges* to Type 5 = *Areas with very favorable conditions*).
  3. `000008IW`: **Låg ekonomisk standard (%)** — Percentage of population living with disposable income below 60% of the national median.

In the cell below, we execute a live REST request to SCB for all 47 sub-areas of Borås, unpack the JSON-stat data cube into a Pandas DataFrame, rank neighborhoods by socioeconomic need, and compute Borås's municipal mean!


In [ ]:
# ==============================================================================
# LIVE REST API INGESTION: STATISTICS SWEDEN (SCB) API v2 (BORÅS SUB-AREAS)
# Endpoint: Table TAB6586 (RegSO Socioeconomic Index) - Public Open Data (No Key)
# ==============================================================================

# Construct query URL targeting all 47 RegSO areas within Borås (municipality 1490)
boras_regso_codes = ",".join([f"1490R{i:03d}" for i in range(1, 48)])
url_scb_boras = (
    f"https://statistikdatabasen.scb.se/api/v2/tables/TAB6586/data?lang=sv"
    f"&valueCodes[Region]={boras_regso_codes}"
    f"&valueCodes[ContentsCode]=000008IT,000008IU,000008IW"
    f"&valueCodes[Tid]=2023"
    f"&codelist[Region]=vs_RegSO2020"
)

print("Connecting to Statistics Sweden (SCB) Open REST API v2...")

try:
    req = urllib.request.Request(url_scb_boras, headers={'User-Agent': 'C1DB1B-Workshop/2026'})
    with urllib.request.urlopen(req, timeout=12) as response:
        raw_scb = json.loads(response.read().decode('utf-8'))
    
    # 1. Unpack JSON-stat dimensions
    reg_labels = raw_scb['dimension']['Region']['category']['label']
    metric_labels = raw_scb['dimension']['ContentsCode']['category']['label']
    time_labels = raw_scb['dimension']['Tid']['category']['label']
    
    records = []
    val_idx = 0
    for r_code, r_name in reg_labels.items():
        clean_name = r_name.replace("Borås (", "").replace(")", "").strip()
        for m_code, m_name in metric_labels.items():
            for t_code, t_name in time_labels.items():
                v = raw_scb['value'][val_idx] if val_idx < len(raw_scb['value']) else None
                records.append({
                    'regso_code': r_code,
                    'sub_area_name': clean_name,
                    'indicator': m_name,
                    'value': v
                })
                val_idx += 1
                
    # 2. Pivot into a clean tabular DataFrame
    df_boras_live = pd.DataFrame(records).pivot(
        index=['regso_code', 'sub_area_name'],
        columns='indicator',
        values='value'
    ).reset_index()
    df_boras_live.columns.name = None
    df_boras_live = df_boras_live.rename(columns={
        'Socioekonomiskt index': 'socioeconomic_index',
        'Områdestyp': 'area_type',
        'Andelen personer med låg ekonomisk standard (oavsett ålder)': 'low_income_share_pct'
    })
    
    print(f" Successfully harvested {len(df_boras_live)} Borås sub-districts from SCB!")
    print("\nTop 5 Borås Neighborhoods by Socioeconomic Challenge Index (SCB 2023):")
    display(df_boras_live.sort_values(by='socioeconomic_index', ascending=False)[
        ['sub_area_name', 'socioeconomic_index', 'low_income_share_pct', 'area_type']
    ].head(5))
    
    boras_mean = round(df_boras_live['socioeconomic_index'].mean(), 2)
    print(f"\n Calculated Borås Municipal Mean Socioeconomic Index: {boras_mean}")
    print("👉 Verification: This exact empirical average (9.24) is integrated into our regional dataset for municipality 1490!")

except Exception as err:
    print(f"ℹ️ Network notice: Could not connect directly to SCB API ({err}).")
    print("Using pre-aggregated municipal records from VGR_External_Health_Socioeconomic.csv.")


---
### Step 2.2: Relational Pipeline Amalgamation (The "4+1" Join)
Having seen how public sector data is sourced from SCB, we now load the comprehensive regional indicators table: `VGR_External_Health_Socioeconomic.csv`.

This dataset covers all **52 municipalities** and contains:
- `municipality_code`: Primary key.
- `municipality_name`: Municipal name.
- `median_income_ksek`: Median annual disposable income in thousands of SEK (kSEK).
- `avg_travel_time_min`: Average transit time (in minutes) to the nearest emergency acute care hospital.
- `scb_socioeconomic_index`: **Official Statistics Sweden (SCB) Socioeconomic Index** aggregated from 650 RegSO sub-areas across VGR (Table TAB6586).
- `care_utilization_rate`: Annual outpatient primary healthcare visits per capita.

#### Relational Join Rule:
We execute an **inner join** on `municipality_code`. In decision-critical systems, we must immediately audit row counts:
$$\text{Row Count}_{	ext{amalgamated}} == \text{Row Count}_{	ext{baseline}} == 52$$
Zero rows must be dropped during the merge.


In [ ]:
# ==============================================================================
# TASK 1 (STUDENT): PERFORM KEY-BASED PIPELINE AMALGAMATION
# ==============================================================================

ext_file = "VGR_External_Health_Socioeconomic.csv"
df_external = pd.read_csv(ext_file)

# TODO: Perform an inner merge between df_baseline and df_external on 'municipality_code'
# Hint: df_amalgamated = pd.merge(df_baseline, df_external, on='municipality_code', how='inner')
df_amalgamated = None  # REPLACE WITH YOUR MERGE CODE

# Audit verification
# TODO: Verify that df_amalgamated has exactly 52 rows


---
## 🧹 Part 3: Data Hygiene & Imputation Sensitivity Analysis

### Decision Theory Grounding: The Concealed Traps of Administrative Data
Administrative records frequently contain non-standard missing data tokens. For instance, Statistics Sweden officially documents:
> *`..` = Data not available, too uncertain to be published, or confidential.*

Other government tables introduce whitespace (` `), single hyphens (`-`), or text placeholders (`N/A`). If loaded directly, Pandas treats these numeric columns as **object/string**, breaking all downstream statistical calculations.

#### The Strategic Risks of Imputation:
When filling missing records in decision support:
1. **Mean Imputation ($\bar{x}$)**: Replaces missing values with the arithmetic mean. While computationally simple, it artificially concentrates data mass at the center, **compressing variance ($\sigma_{\text{mean}} < \sigma_{\text{original}}$)** and falsely understating risk.
2. **Median Imputation ($\tilde{x}$)**: Replaces missing values with the 50th percentile. More robust to extreme skewness, but can systematically distort rankings for peripheral rural communities.


In [ ]:
# ==============================================================================
# STEP 3.1: UNMASKING ADMINISTRATIVE NULL TOKENS & CONVERTING TO NP.NAN
# ==============================================================================

null_markers = ['..', '...', ' - ', '-', ' ', '', 'N/A', 'na']

clean_cols = ['avg_travel_time_min', 'median_income_ksek', 'scb_socioeconomic_index', 'care_utilization_rate']

print("Auditing unmasked missing tokens across indicators:")
for col in clean_cols:
    unmasked = df_amalgamated[col].isin(null_markers).sum()
    print(f"   • {col:<26}: {unmasked} administrative missing tokens detected")
    df_amalgamated[col] = df_amalgamated[col].replace(null_markers, np.nan).astype(float)

print(f"\nTotal NaNs across registry after harmonization:\n{df_amalgamated[clean_cols].isna().sum()}")


---
### Step 3.2: Imputation Sensitivity Experiment (Mean vs. Median)
To demonstrate the mathematical consequences of imputation on capital allocation, we evaluate `avg_travel_time_min` (which contains unmasked administrative NaNs).

We evaluate:
1. **Original Baseline Standard Deviation**: $\sigma_{\text{original}}$ (computed on available observed cases via pairwise deletion).
2. **Mean-Imputed Standard Deviation**: $\sigma_{\text{mean}}$ (filling missing values with $\bar{x}$).
3. **Median-Imputed Standard Deviation**: $\sigma_{\text{median}}$ (filling missing values with $\tilde{x}$).
4. **Variance Compression Metric**:
   $$\Delta \sigma_{\%} = \frac{\sigma_{\text{imputed}} - \sigma_{\text{original}}}{\sigma_{\text{original}}} \times 100$$


In [ ]:
# ==============================================================================
# TASK 2 (STUDENT): EXECUTE IMPUTATION SENSITIVITY EXPERIMENT
# ==============================================================================

target_feature = 'avg_travel_time_min'

# 1. Compute baseline observed statistics (ignoring NaNs)
orig_mean = df_amalgamated[target_feature].dropna().mean()
orig_median = df_amalgamated[target_feature].dropna().median()
orig_std = df_amalgamated[target_feature].dropna().std()

# TODO: Create mean-imputed series
# s_mean = df_amalgamated[target_feature].fillna(orig_mean)
s_mean = None

# TODO: Create median-imputed series
# s_median = df_amalgamated[target_feature].fillna(orig_median)
s_median = None

# TODO: Compute standard deviations of imputed series and print comparison


---
### 🤔 Business Decision Question 1: Imputation Integrity & Rural Equity in Capital Allocation

#### Executive Decision Scenario:
Assume the VGR board utilizes a mathematical allocation formula where **higher standard deviations in travel time trigger emergency funding** for remote rural healthcare transport.

#### Decision Analysis Questions:
1. Explain why mean imputation systematically compresses the standard deviation ($\sigma$).
2. If an analyst naively imputes missing transit times for peripheral municipalities (e.g., Grästorp or Tranemo) using the county mean ($\approx 34\text{ min}$), what strategic and ethical harm is inflicted upon those communities?

---
#### ✍️ Your Analysis & Recommendation for Question 1:
*(Write your detailed analysis and recommendation here...)*


---
## 🎛️ Part 4: Preprocessing Choice Menu & Group Portfolio Demonstration

### Decision Theory Grounding: The Non-Commensurate Scale Dilemma
Our amalgamated registry contains features measured on vastly disparate units:
- `median_income_ksek`: Thousands of SEK (scale: $250$ to $410$)
- `avg_travel_time_min`: Travel duration in minutes (scale: $15$ to $75$)
- `scb_socioeconomic_index`: SCB dimensionless composite index (scale: $4.6$ to $13.0$)
- `care_utilization_rate`: Annual visits per capita (scale: $2.0$ to $4.5$)

If fed raw into distance-based clustering (e.g., K-Means) or linear optimization algorithms, income will dominate by several orders of magnitude purely due to its measurement unit, completely masking critical clinical and travel disparities!

---

### The Preprocessing Choice Menu (Group Portfolio Strategy)
To ensure analytical diversity across student project teams, each group selects and executes **ONE** data-diagnosed transformation tailored to their chosen external variable:

| Option | Preprocessing Technique | Mathematical Condition | Formula | Target Feature |
| :---: | :--- | :--- | :---: | :--- |
| **A** | **Log Transformation** | Severe right skewness (long upper tail) | $x_{\text{new}} = \ln(1 + x)$ | `median_income_ksek` |
| **B** | **Robust Winsorization** | Extreme isolated outliers | $x_{\text{capped}} = \min(x, P_{95})$ | `avg_travel_time_min` |
| **C** | **Standard Z-Score** | Approximately symmetric / bell curve | $z = \frac{x - \mu}{\sigma}$ | `scb_socioeconomic_index` |
| **D** | **Min-Max Feature Scaling**| Bounded $[0, 1]$ interval required | $x_{\text{scaled}} = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$ | `care_utilization_rate` |


In [ ]:
# ==============================================================================
# STEP 4.1: DATA DIAGNOSTIC AUDIT (SKEWNESS, SPREAD & OUTLIER DETECTION)
# ==============================================================================

diag_cols = ['median_income_ksek', 'avg_travel_time_min', 'scb_socioeconomic_index', 'care_utilization_rate']

diag_records = []
for c in diag_cols:
    s = df_amalgamated[c].dropna()
    q25, q75 = s.quantile(0.25), s.quantile(0.75)
    iqr = q75 - q25
    outliers = ((s < (q25 - 1.5 * iqr)) | (s > (q75 + 1.5 * iqr))).sum()
    diag_records.append({
        'Feature': c,
        'Mean': s.mean(),
        'Median': s.median(),
        'StdDev': s.std(),
        'Skewness': s.skew(),
        'IQR': iqr,
        'Outliers_1.5IQR': outliers,
        'Recommended_Option': 'A (Log)' if s.skew() > 1.0 else ('B (Winsor)' if outliers > 0 else ('C (Z-Score)' if abs(s.skew()) < 0.2 else 'D (Min-Max)'))
    })

df_diag = pd.DataFrame(diag_records)
print("=" * 95)
print("FEATURE DISTRIBUTION DIAGNOSTIC MATRIX")
print("=" * 95)
display(df_diag)


### 🛠️ Task 3: Apply Your Assigned Group Portfolio Technique
Uncomment and execute your group's chosen technique below:

In [ ]:
# ==============================================================================
# TASK 3 (STUDENT): APPLY YOUR ASSIGNED GROUP PORTFOLIO TECHNIQUE
# Instructions: Uncomment and execute the ONE technique assigned to your group.
# ==============================================================================

# OPTION A: Log Transformation (For Right-Skewed Data: e.g. median_income_ksek)
# df_amalgamated['income_log1p'] = np.log1p(df_amalgamated['median_income_ksek'])
# print("Applied Option A: Log Transformation to median_income_ksek")

# OPTION B: Robust Winsorization (For Outlier-Heavy Data: e.g. avg_travel_time_min)
# cap_val = df_amalgamated['avg_travel_time_min'].quantile(0.95)
# df_amalgamated['travel_time_winsorized'] = np.where(df_amalgamated['avg_travel_time_min'] > cap_val, cap_val, df_amalgamated['avg_travel_time_min'])
# print(f"Applied Option B: Winsorized avg_travel_time_min at 95th percentile ({cap_val:.2f} min)")

# OPTION C: Standard Z-Score Normalization (For Symmetric Data: e.g. scb_socioeconomic_index)
# mu = df_amalgamated['scb_socioeconomic_index'].mean()
# sig = df_amalgamated['scb_socioeconomic_index'].std()
# df_amalgamated['socioeconomic_zscore'] = (df_amalgamated['scb_socioeconomic_index'] - mu) / sig
# print("Applied Option C: Standard Z-Score to scb_socioeconomic_index")

# OPTION D: Min-Max Feature Scaling (For Bounded [0, 1] Scaling: e.g. care_utilization_rate)
# x_min = df_amalgamated['care_utilization_rate'].min()
# x_max = df_amalgamated['care_utilization_rate'].max()
# df_amalgamated['care_utilization_minmax'] = (df_amalgamated['care_utilization_rate'] - x_min) / (x_max - x_min)
# print("Applied Option D: Min-Max Scaling to care_utilization_rate")


---
### 🤔 Business Decision Question 2: Preprocessing Justification & Non-Commensurate Scale Risks

#### Decision Analysis Questions:
1. State the technique selected by your group (**Option A, B, C, or D**) and cite the mathematical/distributional characteristics of your variable that justified this choice.
2. In regional capital allocation, what unintended distortions could occur if your chosen feature was left unscaled during multi-criteria ranking?

---
#### ✍️ Your Analysis & Recommendation for Question 2:
*(Write your detailed analysis and recommendation here...)*


---
## 🛡️ Part 5: Validation Governance (The 20% Hold-Out Lock)

### Decision Theory & Governance Grounding: Preventing Data Leakage
In data-driven decision-making, an organization faces the critical risk of **model optimism / overfitting**: algorithms perform exceptionally well on data they were trained on, but fail catastrophically when deployed across unobserved operational environments.

#### The Golden Rule of Validation Governance:
Before performing any predictive regression or multi-criteria optimization in **Module 3**, we must partition our dataset and **permanently sequester a 20% hold-out test set**.

```
    ========================================================================================
                          FIREWALL ARCHITECTURE: 80% / 20% PARTITION
    ========================================================================================
    [ Amalgamated Registry (52 Municipalities) ]
                  │
                  ├───► 80% Training Set (`VGR_Training_Model_Set.csv`)  : 41 Municipalities
                  │     (Used for model training, KPI tuning, and weights calibration)
                  │
                  └───► 20% Hold-Out Lock (`VGR_Holdout_Test_Set.csv`)   : 11 Municipalities
                        (LOCKED & SEQUESTERED: Zero parameter tuning or leakage permitted!)
    ========================================================================================
```

#### Zero-Leakage Governance Assertion:
The intersection of primary keys between training and hold-out sets must be strictly null:
$$\mathcal{M}_{\text{train}} \cap \mathcal{M}_{\text{test}} = \emptyset$$


In [ ]:
# ==============================================================================
# TASK 4 (STUDENT): EXECUTE DETERMINISTIC TRAIN / TEST SPLIT & EXPORT SETS
# ==============================================================================

# TODO: Use train_test_split from sklearn.model_selection with test_size=0.20, random_state=42
# df_train, df_test = train_test_split(df_amalgamated, test_size=0.20, random_state=42)
df_train, df_test = None, None

# TODO: Verify zero leakage between train and test municipality codes
# TODO: Export to 'VGR_Training_Model_Set.csv' and 'VGR_Holdout_Test_Set.csv'


---
### 🤔 Business Decision Question 3: Validation Governance & Ethical Safeguards Against Model Optimism

#### Decision Analysis Questions:
1. Why must the 20% hold-out test set be sequestered *before* undertaking predictive modeling in Module 3?
2. If healthcare executives had access to the test set during metric calibration, how might unconscious bias lead to "p-hacking" or premature optimization in capital allocation?

---
#### ✍️ Your Analysis & Recommendation for Question 3:
*(Write your detailed analysis and recommendation here...)*


---
## 🏁 Part 6: Gateway 2 Checklist & Submission Artifacts

Congratulations on completing **Workshop 2: Data Preparation, Hygiene & Pipeline Amalgamation**!

### Verification Checklist Before Canvas / LMS Submission:
- [x] **Amalgamation Audit**: Demographics merged with external indicators with zero row attrition ($52/52$).
- [x] **Live API Comprehension**: Examined how public sector REST APIs (SCB Table TAB6586) deliver open indicators.
- [x] **Hygiene & Imputation Audit**: Detected and replaced administrative missing tokens (`..`, `-`, ` `) with `np.nan`, and quantified variance compression.
- [x] **Preprocessing Choice**: Diagnosed distribution characteristics and executed your group's assigned transformation.
- [x] **Validation Governance**: Sequestered exactly 20% of municipalities into `VGR_Holdout_Test_Set.csv` and confirmed zero primary key leakage.
- [x] **Business Decision Questions**: Answered all three Business Decision Questions.

---
*Course*: C1DB1B – Data Management for Decision-Making (7.5 ECTS)  
*Faculty*: Faculty of Business, Work Life and Social Welfare, University of Borås  
*Instructor*: Gideon Mbiydzenyuy (gideon.mbiydzenyuy@hb.se)
